Universidade Federal Rural do Rio de Janeiro - Instituto Multidisciplinar - Ciência da Computação - 2026.1  
Exercício Prático 1 de Inteligência Artificial  
Nome: Guilherme Oliveira Figueiredo  
Matrícula: 20220011840  

# 1. Definição do Problema de Busca

O problema proposto consiste em modelar a movimentação de um cavalo em um ambiente bidimensional (grid), com o objetivo de determinar se existe um caminho de fuga até a borda do cenário. Caso não exista tal caminho, calculamos a pontuação da área que está acessível ao cavalo.

## Espaço de Estados

O espaço de estados é definido pelas possíveis posições do cavalo no grid. Cada estado pode ser representado por:


(x, y)


onde:
- x representa a linha
- y representa a coluna

## Estado Inicial

O estado inicial será a posição inicial do cavalo no grid.

## Ações

As ações possíveis serão os movimentos do cavalo nas quatro direções:

- cima: ((x-1), y)
- baixo: ((x+1), y)
- esquerda: (x, (y-1))
- direita: (x, (y+1))

Uma ação só é considerada válida se:
- a nova posição estiver dentro dos limites do grid
- o espaço de destino não estiver sendo ocupado por um obstáculo (lagoa/parede)

## Função de Transição

Como as ações já representam diretamente as novas posições, a transição consiste simplesmente em retornar o novo estado:

\[
resultado(s, a) = s'
\]

## Teste de Objetivo

O objetivo é verificar se o cavalo consegue escapar do grid. Um estado é considerado objetivo se a posição do cavalo estiver em qualquer borda do grid, ou seja:

\[
x = 0 ou x = n-1  \ y = 0 ou y = m-1
\]

onde n e m representam o número de linhas e colunas do grid, respectivamente.

## Função de Custo

Cada movimento do cavalo tem custo igual a 1. Assim, o custo total de um caminho é dado pelo número de passos realizados.

## Tipo do Problema

O problema pode ser caracterizado como:

- determinístico
- totalmente observável
- discreto
- com espaço de estados finito
- modelado como um grafo implícito em um grid

## Considerações Adicionais

Caso não exista caminho até a borda do grid, o problema se transforma na determinação da área que fica acessível ao cavalo. Escolhi uma busca em largura para explorar todos os estados alcançáveis a partir do estado inicial.

# 2. Implementação do Ambiente

## Representação do Ambiente

O ambiente foi modelado como uma matriz bidimensional (grid), na qual cada espaço representa uma posição do cenário.   
Utilizando o conjunto de estados e instâncias disponibilizados no moodle:

-  : espaço livre (+1 pt)
- %: lagoa
- +: parede
- J: cereja (+4 pts)
- M: maçã dourada (+11 pts)
- A: enxame de abelhas (-4 pts)
- C: cavalo   

Observação: os espaços com itens em cima também são considerados espaços livres então considero: ponto do espaço + ponto do item. 

Terá uma função mais a frente no relatório que irá transformar esses valores de texto para números para criar os grids que serão usados na implementação das buscas. Os valores do grid serão:

- 0: espaço livre
- 1: lagoa
- 2: cereja
- 3: maça dourada
- 4: enxame de abelhas
- 5: cavalo
- 6: parede 
     
## Representação do Estado  

Um estado do problema é uma possível posição do cavalo no grid, representada dessa forma:  

```python
(x, y)
```
## Classe do Problema

Essa classe contém:

- As dimensões do grid
- O estado inicial = função para encontrar o local cavalo
- Funções para validar posições/estados
- Função de ações
- Função de transição
- Teste do objetivo
  

In [51]:
# Imports
from collections import deque
import heapq

In [52]:
# Classe do Problema
class ProblemaHorse:

    def __init__(self, grid):

        self.grid = grid
        self.linhas = len(grid)
        self.colunas = len(grid[0])
        self.estado_inicial = self.encontrar_cavalo()

    # encontra a posição inicial do cavalo
    def encontrar_cavalo(self):

        for i in range(self.linhas):
            for j in range(self.colunas):

                if self.grid[i][j] == 5:
                    return (i, j)

        return None

    # verifica os limites do grid
    def dentro_do_grid(self, x, y):

        return (
            0 <= x < self.linhas
            and
            0 <= y < self.colunas
        )

    # verifica se tem obstáculo
    def livre(self, x, y):

        return self.grid[x][y] not in [1, 6]

    # ações possíveis
    def acoes(self, estado):

        x, y = estado

        movimentos = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1)
        ]

        validos = []

        for nx, ny in movimentos:

            if (
                self.dentro_do_grid(nx, ny)
                and
                self.livre(nx, ny)
            ):
                validos.append((nx, ny))

        return validos

    # transicao = apenas retorna o estado resultante após a execução da ação
    def transicao(self, estado, acao):
        return acao

    # objetivo = alcançar alguma borda
    def objetivo(self, estado):

        x, y = estado

        return (
            x == 0
            or
            x == self.linhas - 1
            or
            y == 0
            or
            y == self.colunas - 1
        )

### Validação dos Estados

2 funções para validar os possíveis movimentos:  
função "livre" garante que a célula não está ocupada por alguma parede ou lagoa.

### Função de Ações

A função de ações gera os possíveis movimentos do cavalo a partir do estado atual. Os movimentos são: cima, baixo, esquerda e direita, sendo filtrados para apenas aqueles que resultam em posições válidas.

### Função de Transição

Como as ações já representam diretamente os estados de destino, a transição consiste em retornar o próprio estado gerado.

### Teste de Objetivo

O teste de objetivo verifica se o cavalo alcançou a borda do grid, caracterizando uma fuga.  


# 3. Algoritmos de Busca

## Busca em Largura


In [53]:
def bfs(problema):
    fila = deque([(problema.estado_inicial, [])])
    visitados = {problema.estado_inicial} 
    
    nos_expandidos = 0
    nos_descobertos = 1
    profundidade_maxima = 0
    
    while fila:
        estado, caminho = fila.popleft()
        nos_expandidos += 1
        
        # Verifica se atingiu o objetivo (bordas)
        if problema.objetivo(estado):
            final_path = caminho + [estado]
            return {
                "algoritmo": "BFS",
                "caminho": final_path,
                "custo": len(caminho), # Custo é o número de passos
                "nos_expandidos": nos_expandidos,
                "nos_descobertos": nos_descobertos,
                "b": 4, # Fator de ramificação médio
                "d": len(caminho), # Profundidade da solução
                "m": profundidade_maxima
            }

        for prox in problema.acoes(estado):
            if prox not in visitados:
                visitados.add(prox) # Marca aqui para evitar duplicatas na fila
                nos_descobertos += 1
                
                novo_caminho = caminho + [estado]
                profundidade_maxima = max(profundidade_maxima, len(novo_caminho))
                
                fila.append((prox, novo_caminho))

    # Se sair do loop é porque não tem caminho de fuga 
    return {
        "algoritmo": "BFS",
        "caminho": None,
        "nos_expandidos": nos_expandidos,
        "nos_descobertos": nos_descobertos,
        "b": 4,
        "d": None,
        "m": profundidade_maxima,
        "pontuacao": calcular_pontuacao(problema)
    }

## Busca em Profundidade

In [54]:
def dfs(problema):

    pilha = [(problema.estado_inicial, [])]
    visitados = set()
    descobertos = {problema.estado_inicial}

    nos_expandidos = 0
    nos_descobertos = 1
    profundidade_maxima = 0

    while pilha:

        estado, caminho = pilha.pop()

        if estado in visitados:
            continue

        visitados.add(estado)
        nos_expandidos += 1
        profundidade_maxima = max(profundidade_maxima, len(caminho))

        if problema.objetivo(estado):
            return {
                "algoritmo": "DFS",
                "caminho": caminho + [estado],
                "custo": len(caminho),
                "nos_expandidos": nos_expandidos,
                "nos_descobertos": nos_descobertos,
                "b": 4,
                "d": len(caminho),
                "m": profundidade_maxima
            }

        for prox in reversed(problema.acoes(estado)):
            if prox not in visitados and prox not in descobertos:
                pilha.append((prox, caminho + [estado]))
                descobertos.add(prox)
                nos_descobertos += 1

    # Sem caminho de fuga: retorna pontuação
    return {
        "algoritmo": "DFS",
        "caminho": None,
        "nos_expandidos": nos_expandidos,
        "nos_descobertos": nos_descobertos,
        "b": 4,
        "d": None,
        "m": profundidade_maxima,
        "pontuacao": calcular_pontuacao(problema)
    }

## A* com Heurística

A Heurística selecionada foi *distância mais curta até a borda mais próxima*, ou seja ignoramos obstáculos.

In [55]:
# Implementação da heurística

def heuristica(estado, problema):

    x, y = estado

    return min(
        x,
        y,
        problema.linhas - 1 - x,
        problema.colunas - 1 - y
    )

In [56]:
#Implementação da busca

def a_star(problema):
    # (f, g, estado, caminho) -> g incluído para desempatar e facilitar
    h_ini = heuristica(problema.estado_inicial, problema)
    heap = [(h_ini, 0, problema.estado_inicial, [])]
    
    visitados = {} # Guardamos estado: melhor_g_encontrado
    nos_expandidos = 0
    nos_descobertos = 1
    profundidade_maxima = 0

    while heap:
        f, g, estado, caminho = heapq.heappop(heap)

        # Se já achamos um caminho melhor para esse estado, ignora
        if estado in visitados and visitados[estado] <= g:
            continue
            
        visitados[estado] = g
        nos_expandidos += 1
        profundidade_maxima = max(profundidade_maxima, len(caminho))

        if problema.objetivo(estado):
            return {
                "algoritmo": "A*",
                "caminho": caminho + [estado],
                "custo": g,
                "nos_expandidos": nos_expandidos,
                "nos_descobertos": nos_descobertos,
                "b": 4, # Média de sucessores 
                "d": len(caminho), # Profundidade da solução 
                "m": profundidade_maxima
            }

        for prox in problema.acoes(estado):
            novo_g = g + 1
            if prox not in visitados or novo_g < visitados[prox]:
                h = heuristica(prox, problema)
                # Adicionamos novo_g na tupla para o heapq ter como desempatar f
                heapq.heappush(heap, (novo_g + h, novo_g, prox, caminho + [estado]))
                nos_descobertos += 1

    # Sem caminho de fuga: retorna pontuação
    return {
        "algoritmo": "A*",
        "caminho": None,
        "nos_expandidos": nos_expandidos,
        "nos_descobertos": nos_descobertos,
        "b": 4,
        "d": None,
        "m": profundidade_maxima,
        "pontuacao": calcular_pontuacao(problema)
    }

## Calculando Pontuação

Calcula a pontuação da área acessível ao cavalo quando não há fuga, percorrendo todos os espaços alcançáveis a partir do estado inicial usando BFS simples.

In [57]:
def calcular_pontuacao(problema):

    # Valores de pontuação por tipo de célula
    pontos_por_tipo = {
        0: 1,   # espaço livre (+1)
        2: 4,   # cereja  (1 + 3)
        3: 11,  # maçã   (1 + 10)
        4: -4,  # abelhas (1 - 5)
        5: 1,   # cavalo (também é espaço livre, logo: +1)
    }

    visitados = set()
    fila = deque([problema.estado_inicial])
    visitados.add(problema.estado_inicial)
    pontuacao = 0

    while fila:
        x, y = fila.popleft()
        tipo = problema.grid[x][y]
        pontuacao += pontos_por_tipo.get(tipo, 0)

        for nx, ny in problema.acoes((x, y)):
            if (nx, ny) not in visitados:
                visitados.add((nx, ny))
                fila.append((nx, ny))

    return pontuacao

## Convertendo Texto em Matriz para os Testes

In [58]:
# conversor de texto em matriz de inteiros(grid)
def converter_grid(texto):

    simbolos = {
        " ": 0,
        "%": 1,
        "J": 2,
        "M": 3,
        "A": 4,
        "C": 5,
        "+": 6
    }

    linhas = texto.strip("\n").split("\n")

    # lê tamanho
    dimensao = linhas[0]

    linhas_grid = linhas[1:]

    n_linhas, n_colunas = map(
        int,
        dimensao.lower().split("x")
    )

    grid = []

    for linha in linhas_grid:

        nova_linha = []

        # completa linha automaticamente
        linha = linha.ljust(n_colunas)

        for char in linha[:n_colunas]:

            nova_linha.append(
                simbolos.get(char, 0)
            )

        grid.append(nova_linha)

    # completa linhas faltantes (estava dando alguns erros sem isso)
    while len(grid) < n_linhas:

        grid.append([0] * n_colunas)

    return grid

## Função Aplicando os 3 Métodos de Busca

Roda BFS, DFS e A* em uma instância e exibe os resultados comparativos.

In [59]:
def rodar_algoritmos(nome_instancia, mapa_texto):

    grid = converter_grid(mapa_texto)
    problema = ProblemaHorse(grid)

    print(f"{'='*55}")
    print(f"  Instância: {nome_instancia}")
    print(f"  Posição inicial do cavalo: {problema.estado_inicial}")
    print(f"  Dimensões: {problema.linhas}x{problema.colunas}")
    print(f"{'='*55}")

    resultados = []

    for algoritmo, funcao in [("BFS", bfs), ("DFS", dfs), ("A*", a_star)]:

        resultado = funcao(problema)
        resultados.append(resultado)

        print(f"\n--- {algoritmo} ---")

        if resultado["caminho"] is not None:
            print(f"  Situação:        Fuga encontrada!")
            print(f"  Caminho:         {resultado['caminho']}")
            print(f"  Custo (passos):  {resultado['custo']}")
            print(f"  Nós expandidos:  {resultado['nos_expandidos']}")
            print(f"  Nós descobertos: {resultado['nos_descobertos']}")
            print(f"  d (prof. solução): {resultado['d']}")
            print(f"  m (prof. máxima):  {resultado['m']}")
            print(f"  b (ram. médio):    {resultado['b']}")
        else:
            print(f"  Situação:        Sem fuga — cavalo cercado!")
            print(f"  Pontuação:       {resultado['pontuacao']}")
            print(f"  Nós expandidos:  {resultado['nos_expandidos']}")
            print(f"  Nós descobertos: {resultado['nos_descobertos']}")
            print(f"  m (prof. máxima):  {resultado['m']}")

    return resultados

# 4. Testando

## Entice

In [45]:
mapa_1 = """
15x15
%%%% % %   %%%%
%%     %     %%
%  J   % %    %
%  %   %      %
  % %  %   %%  
 %   % %  %  % 
               
%%%%%% C %%%%%%
               
 %   % %   %%A 
   %   %  %%%% 
% %%%% %      %
%  M%  % %%%  %
%%     %     %%
%%%%   %   %%%%
"""

In [46]:
# Rodando a instância Entice
resultados_entice = rodar_algoritmos("Entice", mapa_1)

  Instância: Entice
  Posição inicial do cavalo: (7, 7)
  Dimensões: 15x15

--- BFS ---
  Situação:        Fuga encontrada!
  Caminho:         [(7, 7), (6, 7), (6, 6), (5, 6), (4, 6), (3, 6), (2, 6), (1, 6), (0, 6)]
  Custo (passos):  8
  Nós expandidos:  76
  Nós descobertos: 93
  d (prof. solução): 8
  m (prof. máxima):  8
  b (ram. médio):    4

--- DFS ---
  Situação:        Fuga encontrada!
  Caminho:         [(7, 7), (6, 7), (6, 6), (5, 6), (4, 6), (3, 6), (2, 6), (1, 6), (0, 6)]
  Custo (passos):  8
  Nós expandidos:  9
  Nós descobertos: 18
  d (prof. solução): 8
  m (prof. máxima):  8
  b (ram. médio):    4

--- A* ---
  Situação:        Fuga encontrada!
  Caminho:         [(7, 7), (6, 7), (6, 6), (5, 6), (4, 6), (3, 6), (2, 6), (1, 6), (0, 6)]
  Custo (passos):  8
  Nós expandidos:  50
  Nós descobertos: 88
  d (prof. solução): 8
  m (prof. máxima):  8
  b (ram. médio):    4


## Geometry

In [47]:
mapa_2 = """
30x30
  %%%%%% %%% %%%%% %%%%%% %%  
   %  %  %   %           % %   
%  %  %  %   %   %  %   % %  %
%  %% %  %      %%%  %%% % %% %
%      %  %%%      %      %     
%  %  % %%  %%%%%% %   % %%%%
%%%% %%    % %    % %%%%% %  %
%       % %%%%       %        %
    %%%%%    %%%%%  %   %    %
% %%%        %   %  %  %%%%  %
%   %      %%%    %% %      %  %
%   %  %%%     % %  %  %  %   
%   %  %   %   % %  %% %%%%   
%%%%%%%%    %%%%% %        %%%%
%      %    %     %% %%%   %  %
      %    %   C      %      %
%  % %%%%% %%%%   %   %%%%%%%%
%  %      %    % %%%%%%%   %  %
%% %        %       %  %   %  %
%  % %%%%%%%% %%%%     %      %
%% % %   %  %       %%%%   %  %
   % %      %     %        %%%%
%%%%%%%%%        %  %%% %%%   
%      %    %%%%%%%  %        %
%  %    %   %       %    %   %
%%%%    %   %  %%% %%%%% %   %
   %%% %%% %%   %      % %   %
%  %   % %  %   %  %      %%% %
%       %           %   %      
  %%%%%%  %%%%%%%%%%%  %%% %  
"""

In [61]:
# Rodando a instância Geometry
resultados_geometry = rodar_algoritmos("Geometry", mapa_2)

  Instância: Geometry
  Posição inicial do cavalo: (15, 15)
  Dimensões: 30x30

--- BFS ---
  Situação:        Fuga encontrada!
  Caminho:         [(15, 15), (15, 16), (15, 17), (15, 18), (15, 19), (15, 20), (14, 20), (13, 20), (13, 21), (13, 22), (13, 23), (13, 24), (14, 24), (15, 24), (15, 25), (15, 26), (15, 27), (15, 28), (14, 28), (14, 29)]
  Custo (passos):  19
  Nós expandidos:  204
  Nós descobertos: 225
  d (prof. solução): 19
  m (prof. máxima):  20
  b (ram. médio):    4

--- DFS ---
  Situação:        Fuga encontrada!
  Caminho:         [(15, 15), (14, 15), (14, 16), (14, 17), (15, 17), (16, 17), (16, 16), (17, 16), (18, 16), (18, 15), (18, 14), (17, 14), (17, 13), (17, 12), (17, 11), (18, 11), (18, 10), (18, 9), (17, 9), (17, 8), (17, 7), (18, 7), (18, 6), (18, 5), (17, 5), (17, 4), (16, 4), (15, 4), (14, 4), (14, 3), (14, 2), (15, 2), (16, 2), (17, 2), (18, 2), (19, 2), (20, 2), (21, 2), (21, 1), (21, 0)]
  Custo (passos):  39
  Nós expandidos:  45
  Nós descobertos: 62
 

# 5. Discutindo Resultados, Estatísticas e Particularidades do Algoritmo

## Resultados:
| Instância | Algoritmo | Custo (d) | Nós Expandidos | Nós Descobertos | m |
|-----------|-----------|:---------:|:--------------:|:---------------:|:-:|
| Entice (15x15) | BFS | 8 | 76 | 93 | 8 |
| Entice (15x15) | DFS | 8 | 9 | 18 | 8 |
| Entice (15x15) | A* | 8 | 50 | 88 | 8 |
| Geometry (30x30) | BFS | 19 | 204 | 225 | 20 |
| Geometry (30x30) | DFS | **39** | 45 | 62 | 39 |
| Geometry (30x30) | A* | 19 | **93** | 136 | 19 |

## Análise por Algoritmo  

### - BFS (Busca em Largura)
- Acha o caminho ótimo, pois explora os nós nível por nível. Em ambas as instâncias encontrou o caminho mínimo (custo 8 e 19 respectivamente).
- Só que o custo disso é a memória: BFS foi o algoritmo que mais expandiu e descobriu nós, por ter que manter toda a fronteira em memória. Para o Geometry, expandiu 204 nós para encontrar um caminho de apenas 19 passos.

### - DFS (Busca em Profundidade)
- É o algoritmo mais rápido em nós expandidos — expandiu apenas 9 nós no Entice e 45 no Geometry. Porém, ele não garante o caminho ótimo: no Geometry encontrou um caminho de 39 passos contra o mínimo de 19. Isso porque o DFS segue um único ramo até o fim antes de explorar ramos alternativos, o que faz com que ele dependa da ordem dos movimentos para ter a "sorte" de encontrar o caminho ótimo.
- No Entice, por coincidência, ele acabou encontrando o caminho ótimo de 8 passos.

### - A* com Heurística
- O A* combina o melhor dos dois: garante o caminho ótimo (igual o BFS) e ainda consegue expandir menos nós ao usar a heurística para guiar a busca.
- No Geometry expandiu apenas 93 nós contra 204 do BFS, mantendo o mesmo custo ótimo de 19.
- A heurística utilizada foi a distância mínima de Manhattan até a borda mais próxima, sendo admissível pois nunca superestima o custo real (ignora obstáculos, portanto subestima ou acerta).

### - Parâmetros b, d e m
- O fator de ramificação b é aproximadamente 4 em média (cima, baixo, esquerda, direita), embora na prática seja menor nas bordas e nas adjacências de obstáculos.
- A profundidade da solução d é 8 no Entice e 19 no Geometry para BFS e A*.
- A profundidade máxima explorada m coincide com d no BFS e A* (que param ao encontrar a solução ótima), mas no DFS do Geometry chega a 39, mostrando que teve um longo caminho percorrido antes de atingir a borda.

# 6. Heurística para Problemas com Portais

Com portais, o cavalo pode se teletransportar de um ponto do grid para outro,
o que significa que a distância até a borda pode ser menor do que parece.   
A ideia é adaptar a heurística anterior para levar isso em conta.

Dado o estado atual `(x, y)` e um par de portais `A = (ax, ay)` e `B = (bx, by)`,
calculo três estimativas e pego o menor valor entre elas:

### **1. Caminho direto até a borda** (igual à heurística sem portais):

h_dir(x, y) = min(x, y, (n - 1 - x), (m - 1 - y))

### **2. Caminho passando pelo portal A → B** (chega em A, aparece em B, vai até a borda):

h_portal_AB = manhattan(x, y, ax, ay) + h_dir(bx, by)

### **3. Caminho passando pelo portal B → A** (chega em B, aparece em A, vai até a borda):

h_portal_BA = manhattan(x, y, bx, by) + h_dir(ax, ay)

A heurística final é:

h(x, y) = min(h_dir, h_portal_AB, h_portal_BA)

onde `manhattan(p, q) = |px - qx| + |py - qy|`.

No fim é admissível porque usa a distância Manhattan, que ignora obstáculos e
portanto nunca superestima o custo real. A limitação principal é que não
considera obstáculos no caminho até o portal — em grids muito bloqueados
pode subestimar bastante — mas para fins de admissibilidade isso não é tanto um problema.

fim :) 👍